# Script in which we summarize the properties of the studied galaxies, as well as calculate the proportion of barred galaxies in our sample

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List

## Functions

In [ ]:
def multiStr2Num(n: pd.core.series.Series) -> np.array:
    '''
    Función que convierte una serie de números de formato str a formato float
    (n)  número en formato str
    '''
    Num = np.zeros(len(n))
    for i,c in enumerate(n):
        Num[i] = float(c)
    return Num

In [ ]:
column_names_2 = ['name_1','ra','dec','modelMag_r','petroR90_r','z','MAG_u','MAG_g','MAG_r','MAG_i',
                      'MAG_z','stellarmas','name_2,','k','Q_LSS','eta_k','dnn','d5n','f_LSS']

In [ ]:
def open_txt(fich: str) -> pd.DataFrame:
    '''
    Opens the file and returns a DF.
    (fich) Name of the file to open
    '''
    # Exrtacts line by line
    with open(fich, 'r') as file:
        L = [lines.split() for lines in file]

    # We get the column names from the page
    column_names_2 = ['name_1','ra','dec','modelMag_r','petroR90_r','z','MAG_u','MAG_g','MAG_r','MAG_i',
                      'MAG_z','stellarmas','name_2,','k','Q_LSS','eta_k','dnn','d5n','f_LSS']
    dfo = pd.DataFrame(L[1:], columns=L[0]) # Create de DF

    df = dfo.copy()
    for c in L[0]:
        df[c] = multiStr2Num(df[c])
    return df


def clean(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    '''
    Reduces the columns to the specified in the parametre columns for a given DF
    '''
    #Verifying if the specified columns are in the DataFrame
    valid_columns = [col for col in columns if col in df.columns]

    if len(valid_columns) != len(columns):
        non_valid_columns = set(columns) - set(valid_columns)
        print(f"Warning: Non existent columns have been written,following will not be included: {non_valid_columns}")

    #Croping the DF only with the valid columns

    df_croped = df[valid_columns]

    return df_croped

In [ ]:
def condition(
    serie,
    condicion,
    cumplen='cumplen',
    no_cumplen='no_cumplen'
    
):
    """
    Recibe una serie y una condición booleana (misma longitud).
    Devuelve un DataFrame con:
        - Conteo y porcentaje de cumplimiento y no cumplimiento
        - Media y desviación estándar de la serie completa
        - Nombres de columnas personalizables
    """
    
    assert len(serie) == len(condicion), "La serie y la condición deben tener el mismo tamaño."
    
    total = len(serie)
    cumplen = condicion.sum()
    no_cumplen = total - cumplen

    porcentaje_cumplen = round((cumplen / total) * 100, 2)
    porcentaje_no_cumplen = round((no_cumplen / total) * 100, 2)

    media = round(serie.mean(), 2)
    std = round(serie.std(), 2)

    data = pd.DataFrame({
        'cumplen': [cumplen],
        '%': [porcentaje_cumplen],
        no_cumplen: [no_cumplen],       
        '% no': [porcentaje_no_cumplen],
        'media' : [media],
        'std' : [std]
        
    })

    return data


In [ ]:
def multiple_cond(
    dfs: dict,
    condiciones: dict,
    columna: str,
    nombre_col_cumplen='cumplen',
    nombre_col_no_cumplen='no_cumplen',
    pct_cumplen = 'pct cumplen',
    pct_nocumplen = 'pct_nocumplen'
):
    """
    Recibe:
        - dfs: dictionary for {nombre: DataFrame}
        - condiciones: dictionary ofr {nombre: boolean series} (same length as each DF)
        - columna: name of the numeric column on which to calculate mean and std
    Returns a summarized DF with a row for each data set.
    """

    resumenes = []

    for nombre in dfs:
        df = dfs[nombre]
        condicion = condiciones[nombre]
        serie = df[columna]

        assert len(serie) == len(condicion), f"Condition and DF must hace the same lenght for '{nombre}'"

        total = len(serie)
        cumplen = condicion.sum()
        no_cumplen = total - cumplen

        porcentaje_cumplen = round((cumplen / total) * 100, 2)
        porcentaje_no_cumplen = round((no_cumplen / total) * 100, 2)

        media = round(serie.mean(), 2)
        std = round(serie.std(), 2)

        resumenes.append({
            'data': nombre,
            nombre_col_cumplen: cumplen,
            nombre_col_no_cumplen: no_cumplen,
            pct_cumplen: porcentaje_cumplen,
            pct_nocumplen : porcentaje_no_cumplen,
            'mean': media,
            'std': std
        })

    return pd.DataFrame(resumenes).set_index('data')


In [ ]:
def multiple_mean_std(
    dfs: dict,
    columnas: list,
    decimales: int = 2,
    nombre_filas='data'
):
    """
    Calculates mean ± std for specific columns in multiple DataFrames..

    Parameters:
        - dfs: dict de {nombre: DataFrame}
        - columnas: list of column names to summarize
        - decimales: number of decimals (default 2)
        - nombre_filas: name of the index columns

    Returns:
        - DataFrame in format "mean ± std"
    """

    resumenes = []

    for nombre, df in dfs.items():
        fila = {nombre_filas: nombre}

        for col in columnas:
            valores = df[col].dropna()
            media = round(valores.mean(), decimales)
            std = valores.std()
            n = len(valores)
            sem = round(std / np.sqrt(n), decimales) if n > 0 else np.nan
            fila[col] = f"{media} ± {sem}"

        resumenes.append(fila)

    return pd.DataFrame(resumenes).set_index(nombre_filas)


## Main

In [ ]:
Pann_SIG_AGN = pd.read_csv('data/Pann_SIG_AGN.csv')
SIG_AGN_walls = pd.read_csv('data/SIG_AGN_walls.csv')

SIG_Walls_ALL = pd.read_csv('data/SIG_Walls_ALL.csv')
Pann_SIG_ALL = pd.read_csv('data/Pann_SIG_ALL.csv')

Pann_SIG_ALL.rename(columns={'TType_DS' : 'TType', 'ra_y' : 'ra_z', 'dec_y' : 'dec_z'}, inplace=True)

Pann_SIG_ALL['TType']

In [ ]:
#CONDITIONS

cond = Pann_SIG_AGN['Q_LSS'] > -20
cond2 = SIG_AGN_walls['Q_LSS'] > -20
cond3 = SIG_Walls_ALL['Q_LSS'] > -20
cond4 = Pann_SIG_ALL['Q_LSS'] >-20

In [ ]:
SIG = open_txt('data/SIG.txt')

SIG_morpho = pd.read_csv('data/SIG_TType.csv') #SIG with ObjID and TType



In [ ]:
#DICCIONARIO CON LOS DFS

dfs = {

    'SIG AGN in voids' : Pann_SIG_AGN,
    'SIG AGN in walls' : SIG_AGN_walls,
    'SIG in voids' : Pann_SIG_ALL,
    'SIG in walls' : SIG_Walls_ALL
}

### Morphology 

 Ttype < 0 ---> EARLY TYPE

 Ttype > 0 ---> LATE TYPE

In [ ]:

condiciones_morpho = {

    'SIG AGN in voids' : Pann_SIG_AGN['TType'] > 0,
    'SIG AGN in walls' : SIG_AGN_walls['TType'] > 0,
    'SIG in voids' : Pann_SIG_ALL['TType'] > 0,
    'SIG in walls' : SIG_Walls_ALL['TType'] > 0
}

In [ ]:
Morpho = multiple_cond(dfs, condiciones_morpho, columna = 'TType', nombre_col_cumplen= 'Late Type', nombre_col_no_cumplen= 'Early Type', pct_cumplen= '%_LT', pct_nocumplen= '%_ET')

Morpho

### Star forming rate

log(sSFR) > -0.6173·log(M_{*}) - 4.72 ---> SF

log(sSFR) < -0.6173·log(M_{*}) - 4.72 ---> Q

In [ ]:

condiciones_sSfr = {

    'SIG AGN in voids' : Pann_SIG_AGN['log(sSFR)'] > -0.6173*Pann_SIG_AGN['stellarmass'] - 4.72 ,
    'SIG AGN in walls' : SIG_AGN_walls['log(sSFR)'] > -0.6173*SIG_AGN_walls['stellarmass'] - 4.72,
    'SIG in voids' : Pann_SIG_ALL['log(sSFR)'] > -0.6173*Pann_SIG_ALL['stellarmass'] - 4.72,
    'SIG in walls' : SIG_Walls_ALL['log(sSFR)'] > -0.6173*SIG_Walls_ALL['stellarmass'] - 4.72
}

In [ ]:
Sfr = multiple_cond(dfs, condiciones_sSfr, columna = 'log(sSFR)', nombre_col_cumplen= 'Star forming', nombre_col_no_cumplen='Quenched', pct_cumplen= '%_SF', pct_nocumplen='%_Q')

Sfr

### Color

g - i > 0.2006· \log(M_s) - 1.02 ---> RED

g - i < 0.2006· \log(M_s) - 1.02 ---> BLUE

In [ ]:
Pann_SIG_ALL['g-i'] = Pann_SIG_ALL['MAG_g'] - Pann_SIG_ALL['MAG_i']

condiciones_color = {

    'SIG AGN in voids' : Pann_SIG_AGN['g-i'] > 0.2006* Pann_SIG_AGN['stellarmass'] - 1.02 ,
    'SIG AGN in walls' : SIG_AGN_walls['g-i'] > 0.2006* SIG_AGN_walls['stellarmass'] - 1.02,
    'SIG in voids' : Pann_SIG_ALL['g-i'] > 0.2006* Pann_SIG_ALL['stellarmass'] - 1.02,
    'SIG in walls' : SIG_Walls_ALL['g-i'] > 0.2006* SIG_Walls_ALL['stellarmass'] - 1.02
}

In [ ]:
Color = multiple_cond(dfs, condiciones_color, columna = 'g-i', nombre_col_cumplen= 'RED', nombre_col_no_cumplen= 'BLUE', pct_cumplen= '%_R', pct_nocumplen= '%_B')

Color


### Bar

In [ ]:
DS_Salva = pd.read_csv('data/sigsipsit_lss_230728_DS18.csv')

#We consider a galaxy to be barred if the calculated probability is >= 0.5

Pann_SIG_ALL2 = pd.merge(DS_Salva, Pann_SIG_ALL, on = 'objID', how = 'inner', indicator = 'domde')

Pann_SIG_ALL3 = Pann_SIG_ALL2[Pann_SIG_ALL2['env'] == 'isol']

Pann_SIG_ALL3['P_bar_GZ2']

In [ ]:
dfs_bar = {

    'SIG AGN in voids' : Pann_SIG_AGN,
    'SIG AGN in walls' : SIG_AGN_walls,
    'SIG in voids' : Pann_SIG_ALL3,
    'SIG in walls' : SIG_Walls_ALL
}


condiciones_bar = {

    'SIG AGN in voids' : Pann_SIG_AGN['P_bar_GZ2'] > 0.5,
    'SIG AGN in walls' : SIG_AGN_walls['P_bar_GZ2'] > 0.5,
    'SIG in voids' : Pann_SIG_ALL3['P_bar_GZ2'] > 0.5,
    'SIG in walls' : SIG_Walls_ALL['P_bar_GZ2'] > 0.5
}

In [ ]:
Pann_SIG_ALL.columns #I must add the column "P_bar_GZ2" in order to study the presence of barred galaxies

In [ ]:
Bar = multiple_cond(dfs_bar, condiciones_bar, columna = 'P_bar_GZ2', nombre_col_cumplen= 'With bar', nombre_col_no_cumplen='Without bar', pct_cumplen= '%_with bar', pct_nocumplen= '%_without bar')

#cumple = barred; no cumplen = not barred

Bar

### Table with mean and std values for the properties in df with and without AGN

In [ ]:
dfs_means = {

    'SIG AGN in voids' : Pann_SIG_AGN[cond],
    'SIG AGN in walls' : SIG_AGN_walls[cond2],
    'SIG in voids' : Pann_SIG_ALL[cond4],
    'SIG in walls' : SIG_Walls_ALL[cond3]
}

Resumen_mean_std = multiple_mean_std(dfs_means, columnas= ['Q_LSS', 'g-i', 'dnn', 'log(sSFR)', 'TType', 'stellarmass']) #en voids y walls en general no funciona bien porque esta haciendo la media tb con los -99 pero no se como quitarlos

Resumen_mean_std